# Finetuning Gemini for JAX Code Generation

This notebook demonstrates how to finetune a Gemini model on a custom dataset of JAX-related question-answer pairs. The goal is to create a model that can generate helpful code and explanations for JAX developers.

We will cover the following steps:
1.  **Setup**: Configure the environment and authenticate with the Google AI API.
2.  **Data Loading**: Load and preprocess our custom dataset.
3.  **Finetuning**: Create and run a finetuning job for a Gemini model.
4.  **Testing**: Test the finetuned model with a sample prompt.
5.  **Evaluation**: Run an evaluation to measure the model's performance.

## Imports

First, let's import the necessary libraries. We'll need `google.generativeai` for the core finetuning functionality, along with some helper utilities for data handling and evaluation. We also fetch the repo.

In [1]:
!git clone https://github.com/neel04/pomni.git

Cloning into 'pomni'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 83 (delta 37), reused 65 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 4.22 MiB | 5.38 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [2]:
!pip install uv && uv pip install python-dotenv google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.4/18.4 MB 34.3 MB/s eta 0:00:00
Using Python 3.11.13 environment at: /usr
Resolved 30 packages in 602ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 15ms
Installed 1 package in 6ms
 + python-dotenv==1.1.1


**Make sure to add your envvars! 🙂**

In [3]:
%%writefile /content/.env
GOOGLE_API_KEY=<your_key>

Writing /content/.env


In [4]:
import sys
sys.path.append("/content/pomni/")

import logging
import os
import random
import time
from pathlib import Path
from typing import Optional, TypeVar

from dotenv import load_dotenv
from google.generativeai.generative_models import GenerativeModel
from google.generativeai.models import create_tuned_model, get_tuned_model, list_models

from eval_model import run_async_eval
from utils import (
    LoadedDataset,
    generate_from_model,
    load_finetuned_model,
    split_dataset,
    truncate_sample,
)

## Configuration

This includes loading the Google API key from a `.env` file and registering it to `genai`

In [5]:
import google.generativeai as genai
import os

my_api_key = os.getenv("GOOGLE_API_KEY")

if my_api_key:
    genai.configure(api_key=my_api_key)
else:
    raise ValueError("API key not found. Please set the GOOGLE_API_KEY environment variable.")

In [6]:
os.environ["GOOGLE_API_USE_MTLS_ENDPOINT"] = "never"

assert load_dotenv(), "Couldn't load envvars from .env file"
GOOGLE_API_KEY: Optional[str] = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables.")

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

print("Configuration and logging setup complete.")

Configuration and logging setup complete.


## Data Loading and Preparation

We'll use several datasets containing JAX-related questions and answers, commits, and Stack Overflow discussions. The `LoadedDataset` class from our `utils.py` script helps us load and process these JSON files.

Here, we load three different datasets and then combine them. The data is then split into training and testing sets.

In [7]:
# Load QA pairs from JAX GitHub issues
qa_dataset = (
    LoadedDataset(Path("/content/pomni/data/jax_issues_qa_pairs.json"), truncate_sample)
    .extract_keys("conversation")
    .flatten_on_key("conversation")
    .extract_keys(["text_input", "output"])
)

# Load documented commits from the JAX repository
commits_dataset = LoadedDataset(
    Path("/content/pomni/data/500_documented_commits.json"),
    truncate_sample,
)

# Load QA pairs from Stack Overflow
so_dataset = LoadedDataset(
    Path("/content/pomni/data/so_jax_qa_pairs.json"),
    truncate_sample,
).extract_keys(["text_input", "output"])

# For this tutorial, we'll just use the Stack Overflow dataset for simplicity
# You can experiment with combining datasets:
# combined_dataset = split_dataset(qa_dataset + commits_dataset + so_dataset)
combined_dataset = split_dataset(so_dataset)
train_data, eval_data = combined_dataset["train"], combined_dataset["test"]

print(f"Training data size: {len(train_data)}")
print(f"Evaluation data size: {len(eval_data)}")
print("Sample training data point:")
print(train_data[0])

Training data size: 64
Evaluation data size: 8
Sample training data point:
{'text_input': '<p>I am trying to install jaxlib on my windows 10 by the following command which I found on the documentation..</p>\n<blockquote>\n<p>pip install jaxlib</p>\n</blockquote>\n<p>It shows the following error</p>\n<pre><code>Collecting jaxlib\n  Could not find a version that satisfies the requirement jaxlib (from versions: None)\nNo matching distribution found for jaxlib\n</code></pre>\n', 'output': '<p><em><strong>UPDATE 20240514</strong></em></p>\n<p>Jax is now supported on Windows, because <a href="https://pypi.org/project/jaxlib/#history" rel="nofollow noreferrer">Jaxlib</a> has been so since March 2023.</p>\n<hr>\n<p>Thanks to <a href="https://github.com/cloudhan" rel="nofollow noreferrer">cloudhan</a>\'s <a href="https://github.com/cloudhan/jax-windows-builder" rel="nofollow noreferrer">jax-windows-builder</a>, it is now rather easy to install <a href="https://jax.readthedocs.io/en/latest/" rel

## Model Finetuning

This is the core of our notebook. The `finetune_model` function encapsulates the process of creating and monitoring a finetuning job.

### How it works:
1.  **Select a Base Model**: We search for a base model that supports finetuning (indicated by `createTunedModel` in its supported methods) and is a "flash" model, which is optimized for speed.
2.  **Create a Tuned Model**: We call `create_tuned_model`, passing our training data, a unique ID for our new model, and hyperparameters like `epoch_count`, `batch_size`, and `learning_rate`.
3.  **Monitor the Job**: Finetuning is an asynchronous operation. We poll the status of the job until it's complete.

In [8]:
def finetune_model(data: LoadedDataset, name: str) -> GenerativeModel:
    """
    Finetunes a Gemini model with the provided data.

    Args:
        data: The training dataset.
        name: A unique name for the finetuned model.

    Returns:
        The finetuned GenerativeModel instance.
    """
    # Find a suitable base model for finetuning
    base_models = [
        m
        for m in list_models()
        if "createTunedModel" in m.supported_generation_methods and "flash" in m.name
    ]

    try:
        base_model = base_models[0]
    except IndexError:
        raise ValueError("The API cannot find a model that supports finetuning with 'createTunedModel'. Please use Vertex AI API for finetuning.")

    print(f"Using base model: {base_model.name}")

    # Start the finetuning job
    operation = create_tuned_model(
        source_model=base_model.name,
        training_data=data,
        id=name,
        epoch_count=10,
        batch_size=64,
        learning_rate=9e-3,
    )

    print(f"Finetuning job started. Model Name: {name}")
    print("Waiting for finetuning to complete... This can take a while.")

    # Poll for completion
    while not operation.done():
        time.sleep(60) # Check every minute
        # You can add more detailed progress checks here if needed
        print("Still finetuning...")

    print("Finetuning complete!")

    # Retrieve the finetuned model
    model = get_tuned_model(f"tunedModels/{name}")
    return model

## Run the Finetuning Job

Now, let's run the finetuning process. We'll generate a unique name for our model.

**Note**: This step will take a significant amount of time and will consume API credits. If you have already finetuned a model, you can skip this and load it directly in the next step.

In [9]:
# You can set a specific model name here if you want to reuse it
MODEL_NAME: Optional[str] = None

if MODEL_NAME:
    print(f"Loading existing finetuned model: {MODEL_NAME}")
    finetuned_model = load_finetuned_model(MODEL_NAME)
    finetuned_model_name = MODEL_NAME
else:
    # Create a new model name for this run
    model_name = f"finetuned-jax-expert-{random.randint(0, 10000)}"
    print(f"Starting new finetuning job with model name: {model_name}")
    finetuned_model = finetune_model(train_data, model_name)
    finetuned_model_name = model_name

print(f"Model '{finetuned_model_name}' is ready for use.")

Starting new finetuning job with model name: finetuned-jax-expert-5138


ValueError: The API cannot find a model that supports finetuning with 'createTunedModel'. Please use Vertex AI API for finetuning.

## Test the Finetuned Model

After finetuning, let's test our model with a sample question about JAX. This gives us a quick qualitative check on whether the model has learned from our dataset.

In [ ]:
prompt = "<p>What is a ConcretizationTypeError in JAX?</p>"
print(f"Testing model with prompt: {prompt}")

output = generate_from_model(
    finetuned_model,
    prompt,
    is_str=True,
)

print("\n--- Model Output ---")
print(output)
print("--------------------")

## Evaluate the Model

A single sample is not enough to judge the model's quality. We'll now run a more systematic evaluation using our test set (`eval_data`). The `run_async_eval` function will process the evaluation data and provide metrics on the model's performance.

In [ ]:
print("Running evaluation on the test dataset...")
run_async_eval(logger, eval_data, finetuned_model_name)
print("Evaluation complete. Check the logs for results.")